# **MÓDULO 26 - Projeto Final do Aprofundamento de Analytics**

**Objetivo do Projeto:**

- Tratamento de Dados: Realizar a junção (JOIN) de duas tabelas utilizando SQL para consolidar as informações.
- Análise de Dados: Exportar os dados resultantes para um arquivo CSV.
- Visualização de Dados: Desenvolver um dashboard interativo e informativo para visualização das principais métricas e insights do e-commerce.

**Tabelas Disponibilizadas:**

**Tabela de Transações:** Contém os registros de transações realizadas pelos clientes, incluindo detalhes como ID da transação, valor e outros.


**Tabela de Dados Pessoais:** Contém as informações pessoais dos clientes, como ID do cliente, nome, genero, cidade, etc.

**Chave de Ligação:** As tabelas se relacionam através da coluna ID_CLIENT, que é a chave identificadora dos clientes.

# Etapas do Projeto:

1. Realizar um JOIN SQL nas duas tabelas, unificando as informações através da coluna ID_CLIENT. Você deve justificar a escolha do JOIN (Inner/ Left/ Right ou Full).

2. Exportar os dados consolidados resultantes do JOIN para um arquivo CSV.

3. Utilizar Looker Studio ou Power BI para importar o arquivo CSV.

4. Criar visualizações interativas que apresentem métricas importantes, como total de vendas, número de transações, distribuição geográfica dos clientes, perfil demográfico dos clientes, entre outros.

Abaixo temos a configuração do ambiente SQL:

In [1]:
import sqlite3
import pandas as pd

In [2]:
df_transacoes = pd.read_csv("TB_TRANSACOES_PROJETO_ECOMM.csv", delimiter=';')
df_clientes = pd.read_csv("TB_CLIENTES_PROJETO_ECOMM.csv", delimiter=';')

In [3]:
conn = sqlite3.connect('projeto.db')
# Carregar o DataFrame no banco de dados SQLite - criando tb_transacoes e tb_clientes
df_transacoes.to_sql('tb_transacoes', conn, index=False, if_exists='replace')
df_clientes.to_sql('tb_clientes', conn, index=False, if_exists='replace')

175

In [4]:
# Função para executar consultas SQL e retornar o resultado como um DataFrame
def run_query(query):
    return pd.read_sql_query(query, conn)

In [ ]:
query = """

"""
result_df = run_query(query)
print(result_df)


Para realizar o Join, quero melhor entender e conhecer as bases de dados. Abaixo é possível ver a lista de colunas da tabela de clientes e transaações, respectivamente.


In [11]:
print(df_clientes.columns,"\n", df_transacoes.columns)

Index(['state_name', 'First_name', 'Gender', 'Job_Title', 'Id_client'], dtype='object') 
 Index(['id_client', 'Category', 'Price', 'Card Type'], dtype='object')


O campo em comum, que será nossa chave estrangeira, é id_client. Para clientes ela é uma chave primária, mas não para transações (já que é possível ter mais de uma, ou  nenhuma, transação por cliente).
Isso siginifica que nem todos os id_cliente vão aparecer em transações, e alguns podem aparecer mais de uma vez. Como é possível ver abaixo, COUNT(DISTINCT Id_client) < COUNT(DISTINCT id_client).
Isso indica que, para juntar as tabelas, os dados dos clientes serão repetidos.

In [18]:
query = """
SELECT COUNT(DISTINCT Id_client) FROM tb_clientes
"""
result_df = run_query(query)
print(result_df)
query = """
SELECT COUNT(DISTINCT id_client) FROM tb_transacoes
"""
result_df = run_query(query)
print(result_df)

   COUNT(DISTINCT Id_client)
0                        175
   COUNT(DISTINCT id_client)
0                        241


Então, qual tipo de JOIN deve ser usado? Depende do objetivo que se deseja alcançar. Vamos analisar as consequências de aplicar cada um deles:

**INNER JOIN**
O `INNER` retorna apenas as linhas que têm correspondência em ambas as tabelas. Ou seja, apenas as transações com ID cadastradas *e* que fizeram alguma transação, serão retornadas.
Se o objetivo for analisar apenas as transações feitas por clientes no cadastro aplicado, a tabela de clientes, essa é uma boa opção. Com ela não será possível avaliar se alguma transação foi registrada sem informação do cliente (sem ID válida) e nem identificar quais clientes não realizaram nenhuma transação.

**LEFT JOIN** e **RIGHT JOIN**
O `LEFT` retorna todas as linhas da tabela à esquerda e as correspondentes da tabela à direita. Ou seja, apenas as informações da tabela direita contidas na tabela esquerda. Já o `RIGHT`retorna todas as linhas da tabela à direita e as correspondentes da tabela à esquerda.  Ou seja, apenas as informações da tabela esquerda contidas na tabela da direita. Ambas são equivalentes, a depender de qual tabela estará a esquerda e a direita. Portante, só é necessário analisar um caso.
Considerando a tabela de clientes a esquerda e a de transações a direita, a aplicação do `LEFT JOIN`irá retornar uma tabela com todos os clientes cadastrados na tabela clientes, independente de terem realizado uma transição ou não. Transações realizadas por outros IDs não serão retornadas.
Já na aplicação do `RIGTH JOIN`, o contrária irá ocorrer. Todas as transações serão retornadas com as respectivas informações dos clientes, memso quando não houver ID indicado, ou quando ele não estiver na tabela clientes.
O primeiro caso serve para casos em que deseja-se analisar o comportamento de um conjunto de clientes, tenham feito transações ou não. Já no segundo caso, o foco é a análise de um conjunto de transações, independente de se elas estão relacionadas a umcliente da lista.

**FULL JOIN**
O `FULL` retorna todas as linhas quando há uma correspondência em uma das tabelas. Ou seja, todas as linhas serão retornadas. Essa aplicação é sugerida nos casos em que não se deseja perder nenhuma informação.

**Conclusão**
Como eu não tenho nenhum desses objetivos específicos, e não conheço os bancos de dados fornecidos, vou optar por um `FULL JOIN`. Assim, poderei analisar o comportamento geral dos dados sem perda de informação.


In [27]:
query = """
SELECT * FROM tb_clientes FULL JOIN tb_transacoes ON tb_clientes.Id_client = tb_transacoes.id_client
"""
result_df = run_query(query)
print(result_df)


    state_name First_name  Gender                     Job_Title  Id_client  \
0           TX    Domingo    Male  Structural Analysis Engineer        1.0   
1           MI    Russell    Male            Speech Pathologist        2.0   
2           AL     Kimble    Male           Account Coordinator        3.0   
3           IL   Barnabas    Male               General Manager        4.0   
4           MN     Tanney  Female                  VP Marketing        5.0   
..         ...        ...     ...                           ...        ...   
367       None       None    None                          None        NaN   
368       None       None    None                          None        NaN   
369       None       None    None                          None        NaN   
370       None       None    None                          None        NaN   
371       None       None    None                          None        NaN   

     id_client   Category   Price   Card Type  
0          1.0 

In [26]:
result_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   state_name  301 non-null    object 
 1   First_name  301 non-null    object 
 2   Gender      301 non-null    object 
 3   Job_Title   301 non-null    object 
 4   Id_client   301 non-null    int64  
 5   id_client   296 non-null    float64
 6   Category    296 non-null    object 
 7   Price       296 non-null    object 
 8   Card Type   296 non-null    object 
dtypes: float64(1), int64(1), object(7)
memory usage: 21.3+ KB


Exportando o arquivo como CSV:

In [29]:
result_df.to_csv('dados_ecommerce_final.csv', index=False)

Para a criação do dashboard foi usado o Power BI. Na etapa de transformação de dados substituí todos os dados vazios ou nulos por zero.